# Bài 8 · K-means và khách hàng RFM · Thực hành trên lớp (v3)

Mở notebook từ thư mục bài. Dữ liệu: `data/online_retail_lines.csv.gz` (541.909 dòng hoá đơn của Online Retail, UCI, CC BY 4.0). Mã cho sẵn; việc của bạn là chạy từng ô, đối chiếu với slide và trả lời câu hỏi in đậm.
Không có nhãn đúng: các ô tự kiểm chỉ kiểm phép tính, không kiểm “cụm đúng”.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ROOT = Path.cwd()
dong = pd.read_csv(ROOT / 'data' / 'online_retail_lines.csv.gz', dtype={'InvoiceNo': str, 'StockCode': str, 'Description': str, 'Country': str},
                   keep_default_na=False, na_values={'CustomerID': [''], 'Description': ['']}, parse_dates=['InvoiceDate'])
print(len(dong), 'dòng;', dong.InvoiceDate.min(), '→', dong.InvoiceDate.max())

541909 dòng; 2010-12-01 08:26:00 → 2011-12-09 12:50:00


## Chặng 1 · Từ hoá đơn đến điểm (slide 6–14)

Lọc dòng mua hợp lệ, gom theo khách, tính R (ngày lịch tới 10/12/2011), F (số hoá đơn), M (tổng tiền mua).

In [2]:
hop_le = dong.CustomerID.notna() & (dong.Quantity > 0) & (dong.UnitPrice > 0) & ~dong.InvoiceNo.str.upper().str.startswith('C')
a = dong[hop_le].assign(CustomerID=lambda t: t.CustomerID.astype('int64'), tien=lambda t: t.Quantity * t.UnitPrice)
MOC = pd.Timestamp('2011-12-10')
rfm = a.groupby('CustomerID').agg(lan_cuoi=('InvoiceDate', 'max'), F=('InvoiceNo', 'nunique'), M=('tien', 'sum'))
rfm['R'] = (MOC - rfm.lan_cuoi.dt.normalize()).dt.days
rfm = rfm[['R', 'F', 'M']]
print(len(a), 'dòng mua hợp lệ ·', a.InvoiceNo.nunique(), 'hoá đơn ·', len(rfm), 'khách')
print(rfm.loc[12755])

397884 dòng mua hợp lệ · 18532 hoá đơn · 4338 khách
R     281.00
F       2.00
M    3811.95
Name: 12755, dtype: float64


In [3]:
# TỰ KIỂM · chặng 1
assert len(rfm) == 4338 and len(a) == 397884
assert tuple(rfm.loc[12755, ['R', 'F']]) == (281, 2) and round(rfm.loc[12755, 'M'], 2) == 3811.95
ty_trong = rfm.var() / rfm.var().sum()
print('✅ RFM đúng · tỷ trọng phương sai trên thang gốc:', ty_trong.round(4).to_dict())

✅ RFM đúng · tỷ trọng phương sai trên thang gốc: {'R': 0.0001, 'F': 0.0, 'M': 0.9999}


**Câu hỏi:** trên thang gốc, cột nào quyết định khoảng cách? Sau log1p và chuẩn hoá thì sao? Chạy ô dưới.

In [4]:
sc = StandardScaler().fit(np.log1p(rfm))
X = sc.transform(np.log1p(rfm))
print('tỷ trọng sau log1p + chuẩn hoá:', (X.var(0) / X.var(0).sum()).round(3))
print('trung bình log:', sc.mean_.round(3), '· độ lệch chuẩn log:', sc.scale_.round(3))
assert np.isclose((X ** 2).sum(), len(X) * 3)
print('✅ J với một tâm = n·d =', round((X ** 2).sum(), 2))

tỷ trọng sau log1p + chuẩn hoá: [0.333 0.333 0.333]
trung bình log: [3.857 1.346 6.594] · độ lệch chuẩn log: [1.307 0.683 1.257]
✅ J với một tâm = n·d = 13014.0


## Chặng 2 · Một vòng Lloyd trên sáu điểm (slide 17–27)

Hai hàm: `gan` (giữ tâm, mỗi điểm về tâm gần nhất) và `cap_nhat` (giữ nhóm, tâm = trung bình).

In [5]:
P = np.array([[1, 1], [2, 1], [3, 3], [5, 4], [6, 5], [7, 5]], float)   # An, Bình, Chi, Dung, Hà, Lan

def gan(X, C):
    D = ((X[:, None, :] - C[None, :, :]) ** 2).sum(axis=2)   # d² tới từng tâm
    return D.argmin(axis=1)                                   # hoà → tâm chỉ số nhỏ

def cap_nhat(X, nhan, K):
    return np.array([X[nhan == k].mean(axis=0) for k in range(K)])

def tinh_J(X, nhan, C):
    return float(((X - C[nhan]) ** 2).sum())

C = P[[0, 2]].copy()        # tâm đầu ở An và Chi
day_J = []
nhan = gan(P, C); day_J.append(tinh_J(P, nhan, C))
for _ in range(3):
    C = cap_nhat(P, nhan, 2); day_J.append(tinh_J(P, nhan, C))
    nhan_moi = gan(P, C); day_J.append(tinh_J(P, nhan_moi, C))
    if (nhan_moi == nhan).all(): break
    nhan = nhan_moi
print('J qua từng nửa bước:', np.round(day_J, 4), '· nhóm cuối:', nhan, '· tâm cuối:', C.round(4))

J qua từng nửa bước: [39.     12.     11.625   7.3333  7.3333] · nhóm cuối: [0 0 0 1 1 1] · tâm cuối: [[2.     1.6667]
 [6.     4.6667]]


In [6]:
# TỰ KIỂM · chặng 2
assert np.allclose(day_J[:4], [39, 12, 93 / 8, 22 / 3])
assert all(b <= a + 1e-12 for a, b in zip(day_J, day_J[1:]))
print('✅ khớp slide 26: 39 → 12 → 93/8 → 22/3')

✅ khớp slide 26: 39 → 12 → 93/8 → 22/3


**Câu hỏi:** ở nửa bước nào Chi đổi nhóm, và J giảm bao nhiêu ở nửa bước đó?

## Chặng 3 · Cả thuật toán, chạy trên 4.338 khách (slide 33–43)

In [7]:
def lloyd(X, C0, max_iter=300):
    C = np.array(C0, float); nhan = gan(X, C); doi = []
    for _ in range(max_iter):
        C = cap_nhat(X, nhan, len(C))
        nhan_moi = gan(X, C); doi.append(int((nhan_moi != nhan).sum()))
        if doi[-1] == 0: break
        nhan = nhan_moi
    return nhan, C, doi

ids = [17358, 13497, 12762, 14382]               # 4 khách làm tâm đầu (slide 39–41)
vi_tri = [rfm.index.get_loc(i) for i in ids]
nhan, C, doi = lloyd(X, X[vi_tri])
sk = KMeans(n_clusters=4, init=X[vi_tri], n_init=1, tol=0, algorithm='lloyd').fit(X)
print('số vòng:', len(doi), '· khách đổi cụm 5 vòng đầu:', doi[:5], '· J =', round(tinh_J(X, nhan, C), 2))

số vòng: 31 · khách đổi cụm 5 vòng đầu: [736, 429, 263, 171, 143] · J = 3924.38


In [8]:
# TỰ KIỂM · chặng 3
assert (sk.labels_ == nhan).all() and abs(sk.inertia_ - tinh_J(X, nhan, C)) < 1e-6
assert len(doi) == 31 and doi[0] == 736
print('✅ mã tự viết trùng scikit-learn cho cả', len(nhan), 'khách')

✅ mã tự viết trùng scikit-learn cho cả 4338 khách


**Câu hỏi:** chạy lại với `n_init=20, random_state=42` (mô hình khoá của bài). J nhỏ hơn hay lớn hơn lần chạy trên? Điều đó nói gì về `n_init`?

In [9]:
khoa = KMeans(n_clusters=4, n_init=20, random_state=42, algorithm='lloyd').fit(X)
print('mô hình khoá: J =', round(khoa.inertia_, 2), '· lần chạy trên: J =', round(sk.inertia_, 2))

mô hình khoá: J = 3924.46 · lần chạy trên: J = 3924.38


## Chặng 4 · Chọn K và đọc cụm (slide 46–58)

In [10]:
bang = []
for K in range(2, 7):
    m = KMeans(n_clusters=K, n_init=20, random_state=42, algorithm='lloyd').fit(X)
    bang.append(dict(K=K, inertia=round(m.inertia_, 1), silhouette=round(silhouette_score(X, m.labels_), 4), co_cum=np.bincount(m.labels_).tolist()))
print(pd.DataFrame(bang).to_string(index=False))

 K  inertia  silhouette                         co_cum
 2   6474.1      0.4326                   [1669, 2669]
 3   4856.5      0.3377              [757, 1886, 1695]
 4   3924.5      0.3355         [877, 1579, 720, 1162]
 5   3268.2      0.3182    [1163, 344, 966, 861, 1004]
 6   2840.2      0.3156 [847, 665, 994, 598, 278, 956]


In [11]:
# TỰ KIỂM · chặng 4
assert abs(bang[0]['silhouette'] - 0.4326) < 1e-4 and max(bang, key=lambda r: r['silhouette'])['K'] == 2
print('✅ silhouette K = 2 cao nhất trong K = 2…6 (toàn bộ khách)')
ho_so = rfm.assign(cum=khoa.labels_).groupby('cum').agg(so_khach=('R', 'size'), R=('R', 'median'), F=('F', 'median'), M=('M', 'median'))
print(ho_so.round(1))

✅ silhouette K = 2 cao nhất trong K = 2…6 (toàn bộ khách)
     so_khach      R     F       M
cum                               
0         877   19.0   2.0   450.8
1        1579  183.0   1.0   300.4
2         720    9.0  10.0  3726.1
3        1162   54.0   4.0  1353.7


**Câu hỏi cuối:** viết 3–4 câu tư vấn chọn K = 2 hay K = 4 cho brief “gửi thông điệp khác nhau cho từng nhóm khách”: hai căn cứ có số, một đánh đổi, một phép kiểm cần làm trước khi hành động (gợi ý: thử nghiệm có nhóm đối chứng).